In [1]:
# Import required libraries
import os
import pandas as pd

In [ ]:
# Load the dataset and select relevant columns
df = pd.read_csv('../data/train_participants.tsv', sep='\t')
df = df[['participant_id', 'biological_sex', 'race', 'min_age', 'max_age', 'geolocation', 'arm_name']]
df.head()

In [ ]:
# Feature engineering: Calculate average age and drop original min/max age columns
df['age'] = (df['min_age'] + df['max_age']) / 2
df = df.drop(columns=['min_age', 'max_age'])

# Clean biological_sex column (Standardize Male/Female)
df['biological_sex'] = df['biological_sex'].str.capitalize()

# Clean race column (Group races into specific categories)
race_map = {
    'Black or African American': 'Black',
    'American Indian or Alaska Native': 'American Indian',
    'race: other': 'Other',
    'race: unknown': 'Unknown',
    'Native Hawaiian or Other Pacific Islander': 'Pacific Islander'
}
df['race'] = df['race'].replace(race_map)

# Clean geolocation column (Extract state name only)
df['geolocation'] = df['geolocation'].replace({'US: US: Maryland/Virginia/DC': 'Virginia'})
df['geolocation'] = df['geolocation'].str.replace('US: ', '', regex=False)

# Simplify arm_name to vaccine dose category
def simplify_arm(arm):
    if isinstance(arm, str):
        if 'High Dose' in arm:
            return 'High Dose Fluzone'
        elif 'Standard' in arm or 'standard' in arm:
            return 'Standard Fluzone'
    return 'Other'

df['arm_name'] = df['arm_name'].apply(simplify_arm)

df.head()

In [4]:
# Print unique values for each column to verify cleaning
for col in df.columns:
    print(f"{col}: {df[col].unique()}\n")

participant_id: <StringArray>
[ 'SDY269.SUB112836',  'SDY269.SUB112849',  'SDY269.SUB112854',
  'SDY269.SUB112860',  'SDY269.SUB112881',  'SDY269.SUB112886',
  'SDY269.SUB112846',  'SDY269.SUB112857',  'SDY269.SUB112872',
  'SDY270.SUB112872',
 ...
 'SDY2867.SUB389726',     'EXT100.321P04',     'EXT100.321P05',
     'EXT100.321P11',         'EXT101.Y1',         'EXT101.Y2',
         'EXT101.Y3',         'EXT101.O1',         'EXT101.O2',
         'EXT101.O3']
Length: 3960, dtype: str

biological_sex: <StringArray>
['Female', 'Male']
Length: 2, dtype: str

race: <StringArray>
[           'White',            'Black',            'Asian',
  'American Indian',            'Other',          'Unknown',
 'Pacific Islander',      'Multiracial']
Length: 8, dtype: str

geolocation: <StringArray>
[    'Georgia',           nan, 'Connecticut',       'Texas',  'California',
   'Minnesota',    'Maryland',    'New York',        'Ohio',    'Virginia',
    'Missouri']
Length: 11, dtype: str

age: [28.  39.

In [5]:
# Save the cleaned dataset to the cleaned_data folder
os.makedirs('../cleaned_data', exist_ok=True)
df.to_csv('../cleaned_data/participants_cleaned.csv', index=False)